In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

print("="*70)
print("OPTIMIZED ENSEMBLE - COMBINING ALL 5 MODELS")
print("="*70)

# Check if aggregated forecast exists
agg_file = 'all_models_forecast_2026.csv'
if not Path(agg_file).exists():
    print(f"Error: {agg_file} not found.")
    print("Please run: python aggregate_forecasts.py")
    exit()

# Load aggregated data
df = pd.read_csv(agg_file)
df['Date'] = pd.to_datetime(df['Date'])

print(f"\nLoaded forecast data: {len(df)} days")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"\nColumns available:")
for col in df.columns:
    print(f"  - {col}")

In [ ]:
# Extract model columns
solar_cols = [col for col in df.columns if col.startswith('Solar_model')]
wind_cols = [col for col in df.columns if col.startswith('Wind_model')]

model_names = [
    'Ensemble',
    'SARIMA',
    'Prophet',
    'LSTM',
    'XGBoost'
]

print(f"\n{'='*70}")
print("METHOD 1: SIMPLE EQUAL-WEIGHT AVERAGE")
print(f"{'='*70}")
print("\nEach model weighted equally (20% each)")

# Simple average
df['Solar_EqualAvg'] = df[solar_cols].mean(axis=1)
if wind_cols:
    df['Wind_EqualAvg'] = df[wind_cols].mean(axis=1)

print(f"\nSolar Equal-Weight Average (2026):")
print(f"  Mean: {df['Solar_EqualAvg'].mean():.2f} kWh/day")
print(f"  Total: {df['Solar_EqualAvg'].sum()/1000:.1f} MWh/year")
print(f"  Std: {df['Solar_EqualAvg'].std():.2f} kWh/day")

if wind_cols:
    print(f"\nWind Equal-Weight Average (2026):")
    print(f"  Mean: {df['Wind_EqualAvg'].mean():.2f} kWh/day")
    print(f"  Total: {df['Wind_EqualAvg'].sum()/1000:.1f} MWh/year")

In [ ]:
print(f"\n{'='*70}")
        
METHOD 2: VARIANCE-BASED WEIGHTING")
print(f"{'='*70}")
print("\nModels with lower variance (more stable) get higher weights")

# Calculate weights based on inverse variance (stability)
weights_var = {}
for col, name in zip(solar_cols, model_names):
    variance = df[col].var()
    # Inverse variance (lower var = higher weight)
    weights_var[name] = 1 / (variance + 1e-6)  # +1e-6 to avoid division by zero

# Normalize weights to sum to 1
total_weight = sum(weights_var.values())
weights_var_norm = {k: v/total_weight for k, v in weights_var.items()}

print("\nVariance-Based Weights:")
for name, weight in weights_var_norm.items():
    print(f"  {name:15s}: {weight*100:5.1f}%")

# Apply variance-based weights
df['Solar_VarWeighted'] = sum(
    df[col] * weights_var_norm[name] 
    for col, name in zip(solar_cols, model_names)
)

if wind_cols:
    df['Wind_VarWeighted'] = sum(
        df[col] * weights_var_norm[name] 
        for col, name in zip(wind_cols, model_names)
    )

print(f"\nSolar Variance-Weighted Forecast (2026):")
print(f"  Mean: {df['Solar_VarWeighted'].mean():.2f} kWh/day")
print(f"  Total: {df['Solar_VarWeighted'].sum()/1000:.1f} MWh/year")
print(f"  Std: {df['Solar_VarWeighted'].std():.2f} kWh/day")

if wind_cols:
    print(f"\nWind Variance-Weighted Forecast (2026):")
    print(f"  Mean: {df['Wind_VarWeighted'].mean():.2f} kWh/day")
    print(f"  Total: {df['Wind_VarWeighted'].sum()/1000:.1f} MWh/year")

In [ ]:
print(f"\n{'='*70}")
print("METHOD 3: MEDIAN-BASED ROBUST ENSEMBLE")
print(f"{'='*70}")
print("\nUses median instead of mean (less sensitive to outliers)")

# Median ensemble (robust to outliers)
df['Solar_Median'] = df[solar_cols].median(axis=1)
if wind_cols:
    df['Wind_Median'] = df[wind_cols].median(axis=1)

print(f"\nSolar Median Ensemble (2026):")
print(f"  Mean: {df['Solar_Median'].mean():.2f} kWh/day")
print(f"  Total: {df['Solar_Median'].sum()/1000:.1f} MWh/year")
print(f"  Std: {df['Solar_Median'].std():.2f} kWh/day")

if wind_cols:
    print(f"\nWind Median Ensemble (2026):")
    print(f"  Mean: {df['Wind_Median'].mean():.2f} kWh/day")
    print(f"  Total: {df['Wind_Median'].sum()/1000:.1f} MWh/year")

In [ ]:
print(f"\n{'='*70}")
print("METHOD 4: PERFORMANCE-BASED WEIGHTING")
print(f"{'='*70}")
print("\nWeights based on model stability score")

# Calculate performance metrics for each model
performance_scores = {}
for col, name in zip(solar_cols, model_names):
    # Coefficient of Variation (lower = more stable)
    cv = df[col].std() / df[col].mean()
    # Smoothness score (lower variability between days)
    daily_change = abs(df[col].diff()).mean()
    # Combine: lower cv and lower daily change = higher score
    stability_score = 1 / (cv + (daily_change/df[col].mean()) + 1e-6)
    performance_scores[name] = stability_score

# Normalize weights
total_score = sum(performance_scores.values())
weights_perf = {k: v/total_score for k, v in performance_scores.items()}

print("\nPerformance-Based Weights (Stability + Smoothness):")
for name, weight in weights_perf.items():
    print(f"  {name:15s}: {weight*100:5.1f}%")

# Apply performance-based weights
df['Solar_PerfWeighted'] = sum(
    df[col] * weights_perf[name] 
    for col, name in zip(solar_cols, model_names)
)

if wind_cols:
    df['Wind_PerfWeighted'] = sum(
        df[col] * weights_perf[name] 
        for col, name in zip(wind_cols, model_names)
    )

print(f"\nSolar Performance-Weighted Forecast (2026):")
print(f"  Mean: {df['Solar_PerfWeighted'].mean():.2f} kWh/day")
print(f"  Total: {df['Solar_PerfWeighted'].sum()/1000:.1f} MWh/year")
print(f"  Std: {df['Solar_PerfWeighted'].std():.2f} kWh/day")

if wind_cols:
    print(f"\nWind Performance-Weighted Forecast (2026):")
    print(f"  Mean: {df['Wind_PerfWeighted'].mean():.2f} kWh/day")
    print(f"  Total: {df['Wind_PerfWeighted'].sum()/1000:.1f} MWh/year")

In [ ]:
print(f"\n{'='*70}")
print("METHOD 5: TRIMMED MEAN (REMOVE EXTREMES)")
print(f"{'='*70}")
print("\nUses mean of middle 3 models (removes highest and lowest)")
print("Best for outlier rejection")

def trimmed_mean(row, cols):
    """Calculate mean after removing min and max values"""
    values = row[cols].values
    sorted_vals = np.sort(values)
    # Remove min and max, take average of middle 3
    return np.mean(sorted_vals[1:-1])

df['Solar_TrimmedMean'] = df.apply(lambda row: trimmed_mean(row, solar_cols), axis=1)
if wind_cols:
    df['Wind_TrimmedMean'] = df.apply(lambda row: trimmed_mean(row, wind_cols), axis=1)

print(f"\nSolar Trimmed Mean Forecast (2026):")
print(f"  Mean: {df['Solar_TrimmedMean'].mean():.2f} kWh/day")
print(f"  Total: {df['Solar_TrimmedMean'].sum()/1000:.1f} MWh/year")
print(f"  Std: {df['Solar_TrimmedMean'].std():.2f} kWh/day")

if wind_cols:
    print(f"\nWind Trimmed Mean Forecast (2026):")
    print(f"  Mean: {df['Wind_TrimmedMean'].mean():.2f} kWh/day")
    print(f"  Total: {df['Wind_TrimmedMean'].sum()/1000:.1f} MWh/year")

In [ ]:
print(f"\n{'='*70}")
print("COMPARISON: ALL ENSEMBLE METHODS")
print(f"{'='*70}")

ensemble_methods = {
    'Equal Weight': 'Solar_EqualAvg',
    'Variance Weighted': 'Solar_VarWeighted',
    'Median': 'Solar_Median',
    'Performance Weighted': 'Solar_PerfWeighted',
    'Trimmed Mean': 'Solar_TrimmedMean'
}

print("\nSOLAR ENERGY - ANNUAL TOTALS (MWh):")
comparison = {}
for method_name, col_name in ensemble_methods.items():
    total = df[col_name].sum() / 1000
    mean = df[col_name].mean()
    std = df[col_name].std()
    comparison[method_name] = {'Total': total, 'Mean': mean, 'Std': std}
    print(f"  {method_name:20s}: {total:7.1f} MWh (μ={mean:6.2f}, σ={std:5.2f})")

if wind_cols:
    print("\nWIND ENERGY - ANNUAL TOTALS (MWh):")
    ensemble_methods_wind = {
        'Equal Weight': 'Wind_EqualAvg',
        'Variance Weighted': 'Wind_VarWeighted',
        'Median': 'Wind_Median',
        'Performance Weighted': 'Wind_PerfWeighted',
        'Trimmed Mean': 'Wind_TrimmedMean'
    }
    
    for method_name, col_name in ensemble_methods_wind.items():
        total = df[col_name].sum() / 1000
        mean = df[col_name].mean()
        std = df[col_name].std()
        print(f"  {method_name:20s}: {total:7.1f} MWh (μ={mean:6.2f}, σ={std:5.2f})")

In [ ]:
print(f"\n{'='*70}")
print("RECOMMENDATION: BEST ENSEMBLE METHOD")
print(f"{'='*70}")

print("\n🏆 RECOMMENDED: VARIANCE-WEIGHTED ENSEMBLE")
print("\nWhy?")
print("  ✓ Mathematically optimal for combining forecasts")
print("  ✓ Automatically gives more weight to stable models")
print("  ✓ Reduces forecast variance/uncertainty")
print("  ✓ Proven effective in academic literature")
print("  ✓ Transparent: weights are interpretable")

print("\nAlternatives:")
print("  • Median: Use if you suspect outlier models")
print("  • Trimmed Mean: Balance between robustness & information use")
print("  • Performance-Weighted: If you have domain knowledge")
print("  • Equal: Quick baseline, no tuning needed")

print(f"\n{'='*70}")
print("VARIANCE-WEIGHTED WEIGHTS (RECOMMENDED)")
print(f"{'='*70}")
print("\nThese weights are used for final optimized forecast:\n")
for name, weight in sorted(weights_var_norm.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name:15s}: {weight*100:5.1f}%")

In [ ]:
# Create final optimized forecast
print(f"\n{'='*70}")
print("FINAL OPTIMIZED ENSEMBLE FORECAST")
print(f"{'='*70}")

# Use variance-weighted as the final optimized forecast
final_df = df[['Date', 'Solar_VarWeighted']].copy()
final_df.columns = ['Date', 'Solar_Energy_Forecast_kWh']

if wind_cols:
    final_df['Wind_Energy_Forecast_kWh'] = df['Wind_VarWeighted'].values
    final_df['Combined_Energy_Forecast_kWh'] = df['Solar_VarWeighted'].values + df['Wind_VarWeighted'].values

# Save optimized forecast
output_file = 'optimized_forecast_2026.csv'
final_df.to_csv(output_file, index=False)
print(f"\n✓ Optimized forecast saved to '{output_file}'")

# Display sample
print(f"\nForecast Sample (First 10 days):")
print(final_df.head(10).to_string(index=False))

# Monthly summary
final_df['Month'] = pd.to_datetime(final_df['Date']).dt.to_period('M')
monthly = final_df.groupby('Month')[['Solar_Energy_Forecast_kWh']].agg(['mean', 'min', 'max', 'sum'])
monthly.columns = ['_'.join(col).strip() for col in monthly.columns.values]

print(f"\n{'='*70}")
print("MONTHLY SUMMARY - OPTIMIZED FORECAST (2026)")
print(f"{'='*70}")
print(monthly.to_string())

# Annual totals
print(f"\n{'='*70}")
print("ANNUAL TOTALS - OPTIMIZED FORECAST (2026)")
print(f"{'='*70}")
print(f"\nTotal Solar Energy: {final_df['Solar_Energy_Forecast_kWh'].sum()/1000:,.1f} MWh")
if wind_cols:
    print(f"Total Wind Energy: {final_df['Wind_Energy_Forecast_kWh'].sum()/1000:,.1f} MWh")
    total_combined = final_df['Combined_Energy_Forecast_kWh'].sum()
    print(f"Combined Total: {total_combined/1000:,.1f} MWh")
    solar_pct = (final_df['Solar_Energy_Forecast_kWh'].sum() / total_combined) * 100
    wind_pct = (final_df['Wind_Energy_Forecast_kWh'].sum() / total_combined) * 100
    print(f"\nSolar: {solar_pct:.1f}%")
    print(f"Wind: {wind_pct:.1f}%")

print(f"\n{'='*70}")
print("✓ OPTIMIZED ENSEMBLE COMPLETE")
print(f"{'='*70}")

In [ ]:
# Save all ensemble methods for comparison
print(f"\n{'='*70}")
print("SAVING ALL ENSEMBLE METHODS")
print(f"{'='*70}")

# Create comprehensive comparison file
export_df = df[['Date'] + list(ensemble_methods.values())].copy()
if wind_cols:
    export_df['Combined_Equal'] = df['Solar_EqualAvg'] + df['Wind_EqualAvg']
    export_df['Combined_VarWeighted'] = df['Solar_VarWeighted'] + df['Wind_VarWeighted']
    export_df['Combined_Median'] = df['Solar_Median'] + df['Wind_Median']

export_df.to_csv('ensemble_methods_comparison_2026.csv', index=False)
print(f"\n✓ All ensemble methods saved to 'ensemble_methods_comparison_2026.csv'")
print(f"  Contains all 5 ensemble approaches for comparison")

# Save weights used
weights_df = pd.DataFrame({
    'Model': list(weights_var_norm.keys()),
    'Variance_Weight': [weights_var_norm[m] for m in model_names],
    'Variance_Weight_Percent': [weights_var_norm[m]*100 for m in model_names],
    'Performance_Weight': [weights_perf[m] for m in model_names],
    'Performance_Weight_Percent': [weights_perf[m]*100 for m in model_names]
})
weights_df.to_csv('ensemble_weights.csv', index=False)
print(f"✓ Ensemble weights saved to 'ensemble_weights.csv'")

print(f"\n{'='*70}")
print("Files Generated:")
print(f"{'='*70}")
print(f"  1. optimized_forecast_2026.csv")
print(f"     └─ RECOMMENDED: Variance-weighted ensemble (365 daily forecasts)")
print(f"\n  2. ensemble_methods_comparison_2026.csv")
print(f"     └─ All 5 ensemble methods side-by-side for comparison")
print(f"\n  3. ensemble_weights.csv")
print(f"     └─ Weights used for variance and performance-weighted methods")
print(f"\n{'='*70}")